# IMPORT

In [61]:
import pandas as pd
import numpy as np
from pathlib import Path

# DEFINE FOLDERS

In [62]:
STATION_FEATURE_FOLDER = Path("../data/station_features")
DATASET_INPUT_FOLDER = Path("../data/clean_data_csv")

folders = [DATASET_INPUT_FOLDER, STATION_FEATURE_FOLDER]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# LOAD DATA

In [63]:
df = pd.read_csv(f"{DATASET_INPUT_FOLDER}/final_clean_split_data.csv")

print("(Rows, Columns):", df.shape)
print("Stations:", df["station_code"].nunique())
print(df.columns)

(Rows, Columns): (6710838, 20)
Stations: 1417
Index(['timestamp', 'station_code', 'latitude', 'longitude', 'capacity',
       'total_bikes_now', 'total_bikes_taken', 'temperature', 'wind_speed',
       'wind_direction', 'weather_code', 'is_day', 'hour', 'day_of_week',
       'is_weekend', 'sin_hour', 'cos_hour', 'total_bikes_lag1',
       'total_bikes_lag2', 'split'],
      dtype='object')


# NORMALIZE USAGE

In [64]:
df["usage_ratio"] = df["total_bikes_now"] / df["capacity"]
df["usage_ratio"] = df["usage_ratio"].clip(0, 1)

# DEFINE FEATURE EXTRACTION PER STATION

In [65]:
def extract_station_features(group):
    features = {}

    # Basic usage
    features["mean_usage"] = group["usage_ratio"].mean()
    features["std_usage"] = group["usage_ratio"].std()
    features["min_usage"] = group["usage_ratio"].min()
    features["max_usage"] = group["usage_ratio"].max()

    # Temporal behavior
    features["mean_weekend"] = group[group["is_weekend"] == 1]["usage_ratio"].mean()
    features["mean_weekday"] = group[group["is_weekend"] == 0]["usage_ratio"].mean()

    features["day_usage"] = group[group["is_day"] == 1]["usage_ratio"].mean()
    features["night_usage"] = group[group["is_day"] == 0]["usage_ratio"].mean()

    # Daily amplitude
    hourly_mean = group.groupby("hour")["usage_ratio"].mean()
    features["daily_amplitude"] = hourly_mean.max() - hourly_mean.min()

    # Weather correlation
    for col in [
        "temperature",
        "wind_speed",
        "wind_direction",
        "weather_code"
    ]:
        if group[col].nunique() > 1:
            features[f"corr_{col}"] = group["usage_ratio"].corr(group[col])
        else:
            features[f"corr_{col}"] = 0.0

    # Spatial info
    features["latitude"] = group["latitude"].iloc[0]
    features["longitude"] = group["longitude"].iloc[0]
    features["capacity"] = group["capacity"].iloc[0]

    # NO BIKE LAG
    # features["total_bikes_lag1"] = group["total_bikes_lag1"]
    # features["total_bikes_lag2"] = group["total_bikes_lag2"]

    return pd.Series(features)


# BUILD STATION FEATURE TABLE

In [66]:
station_features = (
    df.groupby("station_code", sort=False, group_keys=False)
    .apply(extract_station_features)
    .reset_index()
)

print("Station feature table shape:", station_features.shape)
station_features.head()

Station feature table shape: (1417, 17)


C:\Users\emirh\AppData\Local\Temp\ipykernel_134492\1119802218.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(extract_station_features)


,station_code,mean_usage,std_usage,min_usage,max_usage,mean_weekend,mean_weekday,day_usage,night_usage,daily_amplitude,corr_temperature,corr_wind_speed,corr_wind_direction,corr_weather_code,latitude,longitude,capacity
0,1001,0.560011,0.281112,0.0,1.000000,0.513568,0.578413,0.654058,0.505620,0.379505,0.018374,0.191405,0.170233,-0.066445,48.857059,2.341798,17.0
1,1002,0.519967,0.300805,0.0,1.000000,0.448492,0.548287,0.614558,0.464379,0.557531,0.130935,0.177805,0.138939,-0.037022,48.857993,2.346917,22.0
2,1006,0.465307,0.228911,0.0,0.939394,0.457781,0.468291,0.427181,0.486030,0.397882,0.050024,0.164551,0.045983,-0.084738,48.862630,2.349807,33.0
3,1007,0.583198,0.268896,0.0,1.000000,0.537458,0.601332,0.712259,0.505740,0.529762,0.263063,0.279246,0.092084,-0.032409,48.863589,2.347779,16.0
4,1008,0.504804,0.300634,0.0,1.000000,0.478562,0.515202,0.627728,0.432812,0.564778,0.225175,0.249439,0.134643,-0.045399,48.863627,2.342624,32.0


# CLEAN FEATURE TABLE

In [67]:
stations_before = set(station_features["station_code"])

print(f"Before cleaning: {station_features.shape}")
station_features = station_features.replace([np.inf, -np.inf], np.nan)
station_features = station_features.dropna()
print(f"After cleaning: {station_features.shape}")

stations_after = set(station_features["station_code"])

dropped_stations = stations_before - stations_after

print(f"Dropped stations: {len(dropped_stations)}")
print(dropped_stations)

Before cleaning: (1417, 17)
After cleaning: (1417, 17)
Dropped stations: 0
set()


# SAVE STATION FEATURES

In [68]:
station_features.to_csv(f"{STATION_FEATURE_FOLDER}/station_features.csv", index=False)
print("Saved station_features.csv")

Saved station_features.csv
